# Network Resilience Walkthrough

This notebook builds a small network, scores redundancy, evaluates an outage, previews the staged restoration output, and renders a simple recovery chart for GitHub viewing.

In [1]:
from pathlib import Path
from geoprompt.network import (
    build_network_graph,
    outage_impact_report,
    restoration_sequence_report,
    supply_redundancy_audit,
)

output_dir = Path.cwd() / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

graph = build_network_graph(
    [
        {"edge_id": "e1", "from_node": "SRC", "to_node": "A", "cost": 1.0},
        {"edge_id": "e2", "from_node": "A", "to_node": "B", "cost": 1.0},
        {"edge_id": "e3", "from_node": "B", "to_node": "C", "cost": 1.0},
        {"edge_id": "tie", "from_node": "SRC", "to_node": "C", "cost": 2.0},
    ],
    directed=False,
)
demand_by_node = {"A": 20.0, "B": 50.0, "C": 10.0}
redundancy = supply_redundancy_audit(graph, source_nodes=["SRC", "C"], demand_by_node=demand_by_node, critical_nodes=["B"])
outage = outage_impact_report(graph, source_nodes=["SRC"], failed_edges=["e1", "e2"], demand_by_node=demand_by_node, customer_count_by_node={"A": 80, "B": 120, "C": 40}, critical_nodes=["B"], outage_hours=2.0)
restoration = restoration_sequence_report(graph, source_nodes=["SRC"], failed_edges=["e1", "e2"], demand_by_node=demand_by_node, critical_nodes=["B"])
print("Top redundancy row:", redundancy[0])
print("Outage summary:", outage)
print("First restoration stage:", restoration["stages"][0])

Top redundancy row: {'node': 'B', 'best_source': 'C', 'source_path_count': 2, 'primary_cost': 1.0, 'backup_cost': 2.0, 'single_source_dependency': False, 'is_critical': True, 'demand': 50.0, 'resilience_tier': 'high', 'priority_score': 51.0}
Outage summary: {'impacted_node_count': 1, 'impacted_nodes': ['A'], 'impacted_customer_count': 80, 'impacted_demand': 20.0, 'critical_node_count': 0, 'critical_impacted_nodes': [], 'outage_hours': 2.0, 'customer_hours_interrupted': 160.0, 'demand_hours_interrupted': 40.0, 'estimated_cost': 1400.0, 'severity_score': 28.0, 'severity_tier': 'medium', 'failed_edge_count': 2}
First restoration stage: {'step': 1, 'repair_edge_id': 'e1', 'priority_score': 1080.0, 'restored_node_count': 1, 'restored_nodes': ['A'], 'restored_demand': 20.0, 'cumulative_restored_demand': 20.0, 'served_node_count': 4, 'remaining_outage_count': 0, 'protected_critical_count': 1}


In [2]:
from geoprompt.tools import build_resilience_summary_report, export_resilience_summary_report

report = build_resilience_summary_report(redundancy, outage_report=outage, restoration_report=restoration)
written = export_resilience_summary_report(report, output_dir / "network-walkthrough.html")
print("Wrote:", written)
print("Available keys:", list(report.keys()))

Wrote: D:\Github\geoprompt\outputs\network-walkthrough.html
Available keys: ['metadata', 'summary', 'redundancy_rows', 'outage_report', 'restoration_report']


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

steps = [stage['step'] for stage in restoration['stages']]
restored = [stage['cumulative_restored_demand'] for stage in restoration['stages']]

plt.rcParams.update({
    'figure.facecolor': '#f8fafc',
    'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#e2e8f0',
    'axes.grid': True,
    'grid.color': '#f1f5f9',
    'grid.linewidth': 0.8,
    'font.family': 'sans-serif',
    'font.size': 10,
})
fig, ax = plt.subplots(figsize=(7, 4))

ax.fill_between(steps, restored, alpha=0.15, color='#22c55e', zorder=2)
ax.plot(steps, restored, marker='o', linewidth=2.5, color='#16a34a',
        markerfacecolor='#ffffff', markeredgecolor='#16a34a', markeredgewidth=2, markersize=8, zorder=3)

for s, r in zip(steps, restored):
    ax.annotate(f'{r:.0f}', (s, r), textcoords='offset points', xytext=(0, 12),
                ha='center', fontsize=9, fontweight='600', color='#166534')

ax.set_title('Cumulative Restored Demand by Repair Step', fontsize=14, fontweight='bold', color='#0f172a', pad=14)
ax.set_xlabel('Repair step', fontsize=10, color='#334155')
ax.set_ylabel('Restored demand', fontsize=10, color='#334155')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(colors='#64748b')
ax.set_ylim(0, max(restored) * 1.2 if restored else 1)
fig.tight_layout()

image_path = output_dir / 'network-restoration-curve.png'
fig.savefig(image_path, dpi=180, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.close(fig)
display(Image(filename=str(image_path)))
print('Saved:', image_path)